# Inference - Q8 model (3-stage U-Net, 3-channel input)
Notebook: `DL-22f2001613-notebook-t32026` (the main submission notebook; mirrored as `notebooks/final_notebook.ipynb` on branch `milestone-2`).

Loads `best_unet_q8.pt`, predicts the four defect masks for every test image, converts them to RLE and writes `/kaggle/working/submission.csv` in the layout of `sample_submission.csv`.

## Setup and settings

In [1]:
import os, glob
import numpy as np
import pandas as pd

IMG_H, IMG_W = 128, 800
DEFECTS = ["defect_a", "defect_b", "defect_c", "defect_d"]
TAU = 0.5                      # probability threshold: raw baseline, tuned later in the analysis step
BATCH_SIZE = 32

def find_file(name, known=None):
    if known and os.path.exists(known):
        return known
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]

SAMPLE_CSV = find_file("sample_submission.csv", "/kaggle/input/competitions/industrial-surface-defect-detection/public/sample_submission.csv")
CKPT = find_file("best_unet_q8.pt", "/kaggle/input/notebooks/mruniksurya/milestone-2-q8/best_unet_q8.pt")
print("sample:", SAMPLE_CSV); print("checkpoint:", CKPT)

sample: /kaggle/input/competitions/industrial-surface-defect-detection/public/sample_submission.csv
checkpoint: /kaggle/input/notebooks/mruniksurya/milestone-2-q8/best_unet_q8.pt


## 1. RLE encoder
1-based start, column-major order; an empty mask is written as a single space, as in `sample_submission.csv`.

In [2]:
def rle_encode(mask):
    flat = np.asarray(mask).flatten(order="F")
    padded = np.concatenate([[0], flat, [0]])
    k = np.where(padded[1:] != padded[:-1])[0] + 1
    return " ".join(f"{s} {e - s}" for s, e in zip(k[0::2], k[1::2]))

## 2. Test Dataset

In [ ]:
import cv2
import torch
from torch.utils.data import Dataset, DataLoader

def find_image_dir(sample_id):
    """Folder that contains a known image file (search under /kaggle/input)."""
    hits = glob.glob(f"/kaggle/input/**/{sample_id}.*", recursive=True) or glob.glob(f"./**/{sample_id}.*", recursive=True)
    if not hits:
        raise FileNotFoundError(f"image {sample_id} not found")
    return os.path.dirname(hits[0])

class TestImages(Dataset):
    """Same preprocessing as training (Paradigm 1): BGR->RGB, /255, channels first. Returns (image, index)."""
    def __init__(self, ids, img_dir):
        self.ids, self.img_dir = list(ids), img_dir
        self.files = {os.path.splitext(f)[0]: f for f in os.listdir(img_dir)}
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        bgr = cv2.imread(os.path.join(self.img_dir, self.files[self.ids[i]]), cv2.IMREAD_COLOR)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        x = torch.from_numpy(rgb.astype(np.float32) / 255.0).permute(2, 0, 1).contiguous()
        return x, i

## 3. Model (identical architecture to `milestone-2-q8`)

In [3]:
import torch.nn as nn


class DoubleConv(nn.Module):
    """(Conv3x3 -> BatchNorm -> ReLU) x 2: the basic U-Net building block."""
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.block(x)

class UNet3Stage(nn.Module):
    """U-Net with 3 down stages + bottleneck. Input (B,in_ch,128,800) -> logits (B,4,128,800)."""
    def __init__(self, in_ch=3, out_ch=4, base=32):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base * 4, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = DoubleConv(base * 8, base * 4)        # input = upsampled (4b) + skip (4b)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)
        self.head = nn.Conv2d(base, out_ch, 1)            # 1x1 conv: 4 independent logits per pixel

    def forward(self, x):
        e1 = self.enc1(x)                                 # 128 x 800
        e2 = self.enc2(self.pool(e1))                     #  64 x 400
        e3 = self.enc3(self.pool(e2))                     #  32 x 200
        b = self.bottleneck(self.pool(e3))                #  16 x 100
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))   # skip connection: reuse fine detail from the encoder
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)                              # raw logits (no sigmoid: the loss applies it)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet3Stage(in_ch=3, out_ch=4, base=32).to(DEVICE)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()
print("loaded", f"{count_params(model):,}", "parameters on", DEVICE)

NameError: name 'torch' is not defined

## 4. Predict the test set

In [5]:
sample = pd.read_csv(SAMPLE_CSV)
test_ids = sample["image_id"].drop_duplicates().tolist()
img_dir = find_image_dir(test_ids[0])
print("test images:", len(test_ids), "| folder:", img_dir)

loader = DataLoader(TestImages(test_ids, img_dir), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

rle_lookup = {}
with torch.no_grad():
    for x, idx in loader:
        probs = torch.sigmoid(model(x.to(DEVICE)))                    # (B,4,128,800)
        preds = (probs > TAU).cpu().numpy().astype(np.uint8)
        for b, i in enumerate(idx.tolist()):
            for k, d in enumerate(DEFECTS):                           # channel order = a,b,c,d as in training
                r = rle_encode(preds[b, k])
                rle_lookup[(test_ids[i], d)] = r if r else " "
print("predicted pairs:", len(rle_lookup))

NameError: name 'find_image_dir' is not defined

## 5. Write the submission and sanity-check it

In [ ]:
sub = sample.copy()
sub["mask_rle"] = [rle_lookup[(i, d)] for i, d in zip(sub["image_id"], sub["defect_type"])]
sub.to_csv("/kaggle/working/submission.csv", index=False)

assert len(sub) == len(sample) == 12000 and list(sub.columns) == list(sample.columns)
empty = sub["mask_rle"].str.strip().eq("")
print("rows:", len(sub), "| overall empty rate:", round(empty.mean(), 4))
print("empty rate per class:\n", empty.groupby(sub["defect_type"]).mean().round(3).to_string())
print("train positive rates for reference: a 9.5%, b 2.7%, c 36%, d 15%  (so expected empty rates are about 0.90, 0.97, 0.64, 0.85)")
sub.head()